In [1]:
import numpy as np
from scipy.stats import f_oneway, f

<div style="background-color:#eef3fb; border-left:5px solid #4a7abf; padding:14px 18px; border-radius:6px; margin-bottom:8px;">

📌 **Question 1 : One-Way ANOVA (Fertilizer Yield)**

Three fertilizers tested on crop yield (tons/acre):

- A: `[5.2, 5.8, 5.5, 6.0, 5.7]`
- B: `[6.1, 6.4, 6.2, 6.8, 6.3]`
- C: `[4.8, 5.0, 4.9, 5.3, 5.1]`

Run one-way ANOVA at α = 0.05. Compute SS_between, SS_within, MS, F, and state your conclusion. Compute η² as well.

</div>

#### Step 1 : Hypotheses

Let $\mu_A, \mu_B, \mu_C$ = true mean crop yields for fertilizers A, B, and C.

$$H_0: \mu_A = \mu_B = \mu_C \quad \text{(all group means are equal)}$$
$$H_1: \text{at least one mean differs}$$

#### Step 2 : Why This Test

*We have three independent groups and want to compare their means simultaneously. Running multiple t-tests would inflate the Type I error rate, so one-way ANOVA is the right tool. It partitions total variability into between-group variance (signal) and within-group variance (noise), and compares them via the F ratio. Population variances are assumed equal across groups (homoscedasticity).*

#### Step 3 : Test Statistic

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}, \qquad MS = \frac{SS}{df}$$

$$SS_{\text{between}} = \sum_{j} n_j (\bar{x}_j - \bar{x}_{\text{grand}})^2 \qquad SS_{\text{within}} = \sum_{j} \sum_{i} (x_{ij} - \bar{x}_j)^2$$

In [2]:
A = np.array([5.2, 5.8, 5.5, 6.0, 5.7])
B = np.array([6.1, 6.4, 6.2, 6.8, 6.3])
C = np.array([4.8, 5.0, 4.9, 5.3, 5.1])

alpha = 0.05
k     = 3                          # number of groups
n     = len(A)                     # observations per group
N     = n * k                      # total observations

In [3]:
mu_A = np.mean(A)
mu_B = np.mean(B)
mu_C = np.mean(C)
mu   = np.mean(np.concatenate([A, B, C]))

print(f'mu_A : {mu_A:.4f}')
print(f'mu_B : {mu_B:.4f}')
print(f'mu_C : {mu_C:.4f}')
print(f'mu   : {mu:.4f}')

mu_A : 5.6400
mu_B : 6.3600
mu_C : 5.0200
mu   : 5.6733


In [4]:
ss_bw     = n * ((mu_A - mu)**2 + (mu_B - mu)**2 + (mu_C - mu)**2)
ss_within = np.sum((A - mu_A)**2) + np.sum((B - mu_B)**2) + np.sum((C - mu_C)**2)
ss_total  = ss_bw + ss_within

print(f'ss_bw    : {ss_bw:.4f}')
print(f'ss_within: {ss_within:.4f}')
print(f'ss_total : {ss_total:.4f}')

ss_bw    : 4.4973
ss_within: 0.8120
ss_total : 5.3093


In [5]:
df_bw     = k - 1
df_within = N - k

ms_bw     = ss_bw / df_bw
ms_within = ss_within / df_within
f_stat    = ms_bw / ms_within

print(f'df_bw    : {df_bw}')
print(f'df_within: {df_within}')
print(f'ms_bw    : {ms_bw:.4f}')
print(f'ms_within: {ms_within:.4f}')
print(f'f_stat   : {f_stat:.4f}')

df_bw    : 2
df_within: 12
ms_bw    : 2.2487
ms_within: 0.0677
f_stat   : 33.2315


#### Step 4 : Critical Value Method

In [6]:
f_crit = f.ppf(1 - alpha, df_bw, df_within)

print(f'f_stat : {f_stat:.4f}')
print(f'f_crit : {f_crit:.4f}')

f_stat : 33.2315
f_crit : 3.8853


**F_stat (33.23) > F_crit (3.885) → Reject H₀**

#### Step 5 : p-value Method

In [7]:
pval = f.sf(f_stat, df_bw, df_within)
print(f'p-val : {pval:.8f}')

p-val : 0.00001280


**p-val (0.00001280) < 0.05 → Reject H₀**

#### Step 6 : Effect Size (η²)

*η² (eta-squared) measures what proportion of total variance is explained by the group factor. It is the ANOVA equivalent of R². Values above 0.14 are conventionally considered large.*

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

In [8]:
eta_sq = ss_bw / ss_total
print(f'eta_sq : {eta_sq:.4f}')

eta_sq : 0.8471


#### Step 7 : Verification with scipy

In [9]:
f_stat_, pval_ = f_oneway(A, B, C)
print(f'f_stat (scipy) : {f_stat_:.4f}')
print(f'p-val  (scipy) : {pval_:.8f}')

f_stat (scipy) : 33.2315
p-val  (scipy) : 0.00001280


#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Grand mean | 5.6733 tons/acre |
| SS_between | 4.4973 |
| SS_within | 0.8120 |
| SS_total | 5.3093 |
| df (between, within) | 2, 12 |
| MS_between | 2.2487 |
| MS_within | 0.0677 |
| F statistic | 33.2315 |
| F critical (α = 0.05) | 3.8853 |
| p-value | 0.00001280 |
| η² | 0.8471 |
| Decision | **Reject H₀** |

Strong evidence that at least one fertilizer produces a different mean crop yield. The F statistic (33.23) far exceeds the critical value (3.89), and the p-value is well below 0.05. η² ≈ 0.847 means the fertilizer choice explains about 85% of the total variance in yield, which is a very large effect. Fertilizer B consistently outperforms both A and C, while C yields the least.

<div style="background-color:#eef3fb; border-left:5px solid #4a7abf; padding:14px 18px; border-radius:6px; margin-bottom:8px;">

📌 **Question 2 : ANOVA Table Interpretation**

Given: SS_between = 120, SS_within = 180, k = 4 groups, N = 24 total observations.

- (a) Fill in the ANOVA table: df, MS_between, MS_within, F.
- (b) Is the result significant at α = 0.05?
- (c) Compute η².

</div>

#### Step 1 : Hypotheses

Let $\mu_1, \mu_2, \mu_3, \mu_4$ = true means for the four groups.

$$H_0: \mu_1 = \mu_2 = \mu_3 = \mu_4 \quad \text{(all group means are equal)}$$
$$H_1: \text{at least one mean differs}$$

#### Step 2 : Why This Test

*The SS values are given directly rather than raw data, so we reconstruct the ANOVA table from first principles. The logic is identical to Question 1: we compare between-group variance to within-group variance via the F ratio, using the same assumptions of independence, normality, and equal variances.*

#### Step 3 : Test Statistic

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}, \qquad MS_{\text{between}} = \frac{SS_{\text{between}}}{k - 1}, \qquad MS_{\text{within}} = \frac{SS_{\text{within}}}{N - k}$$

In [10]:
ss_bw     = 120
ss_within = 180
k         = 4
N         = 24
alpha     = 0.05

In [11]:
df_bw     = k - 1
df_within = N - k
ss_total  = ss_bw + ss_within

ms_bw     = ss_bw / df_bw
ms_within = ss_within / df_within
f_stat    = ms_bw / ms_within

print(f'df_bw    : {df_bw}')
print(f'df_within: {df_within}')
print(f'ms_bw    : {ms_bw:.4f}')
print(f'ms_within: {ms_within:.4f}')
print(f'f_stat   : {f_stat:.4f}')

df_bw    : 3
df_within: 20
ms_bw    : 40.0000
ms_within: 9.0000
f_stat   : 4.4444


#### Step 4 : Critical Value Method

In [12]:
f_crit = f.ppf(1 - alpha, df_bw, df_within)

print(f'f_stat : {f_stat:.4f}')
print(f'f_crit : {f_crit:.4f}')

f_stat : 4.4444
f_crit : 3.0984


**F_stat (4.444) > F_crit (3.098) → Reject H₀**

#### Step 5 : p-value Method

In [13]:
pval = f.sf(f_stat, df_bw, df_within)
print(f'p-val : {pval:.6f}')

p-val : 0.015063


**p-val (0.015063) < 0.05 → Reject H₀**

#### Step 6 : Effect Size (η²)

*η² quantifies the proportion of total variance attributable to group differences.*

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

In [14]:
eta_sq = ss_bw / ss_total
print(f'eta_sq : {eta_sq:.4f}')

eta_sq : 0.4000


#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| SS_between | 120 |
| SS_within | 180 |
| SS_total | 300 |
| df (between, within) | 3, 20 |
| MS_between | 40.0 |
| MS_within | 9.0 |
| F statistic | 4.4444 |
| F critical (α = 0.05) | 3.0984 |
| p-value | 0.015063 |
| η² | 0.4000 |
| Decision | **Reject H₀** |

The result is significant at α = 0.05. F (4.44) exceeds the critical value (3.10) and the p-value (0.015) falls below the threshold, so we reject the null hypothesis. η² = 0.40 indicates that group membership explains 40% of the total variance, which is a large effect by conventional standards.